# Module 5: Dynamic Swarm (15 min)

Apply **Pattern 4** from the deck: agents hand off autonomously — no fixed path, no orchestrator. The route through the team **emerges at runtime** based on what each agent decides to do next.

```
Task (brief)
     │
     ▼
┌──────────────┐        SHARED CONTEXT / WORKING MEMORY
│  Researcher  │  ──────────────────────────────────────
│  (entry)     │  ← uses tools, builds research context
└──────┬───────┘
       │  handoff_to_agent("analyst")   ← autonomous decision
       ▼
┌──────────────┐
│   Analyst    │  ← receives shared context + handoff message
└──────┬───────┘
       │  handoff_to_agent("writer")    ← autonomous decision
       ▼
┌──────────────┐
│    Writer    │  ← produces final memo; decides NOT to hand off
└──────────────┘
```

The path above is not programmed — each agent **reads the task, reads available peers, and decides** who to call next. A different task or different agent descriptions could produce a different path.

**When to use this pattern:**
- Path can't be known in advance
- Benefits from diverse specialist perspectives
- Exploration, brainstorming, multi-domain incident response

**Key Strands API:** `Swarm([agents], entry_point=agent, max_handoffs=..., max_iterations=...)`

**Agent `description` is the routing signal** — agents read peer descriptions to decide who to hand off to.

**Prerequisites:** Modules 1–4 — this module reuses Module 1 tools.

## Tools and Agents in This Module

| Component | Type | Description (used for routing) |
|-----------|------|-------------------------------|
| `get_company_data` | Tool | NovaCart financial/operational data |
| `get_market_benchmarks` | Tool | E-commerce industry benchmarks |
| `get_competitor_data` | Tool | Competitor premium tier details |
| `researcher` | Swarm agent | Market research specialist with tools for company data, benchmarks, and competitor intelligence |
| `analyst` | Swarm agent | Business analyst who evaluates options A/B/C with structured analysis |
| `writer` | Swarm agent | Writes the final executive decision memo |

> **Why `description` matters:** The `description` field is what agents read to decide who to hand off to. Write it for the model, not for humans — be explicit about what the agent can do and when to involve it.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1 — Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2 — Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3 — Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4 — Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")
print("✅ Setup complete!")

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), "..", "01-strands-foundations"))

from strands import Agent
from strands.multiagent import Swarm
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

---

## Part 1 — Create Swarm Agents

Each agent has a `description` — this is the routing signal. When an agent needs to hand off, it reads the descriptions of all peers and decides who is best suited for the next step.

The `system_prompt` tells each agent what to do when it receives a task. The last agent (`writer`) is told **not** to hand off — it produces the final output.

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Options:
  Option A — Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B — Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C — Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

researcher = Agent(
    name="researcher",
    description=(
        "Market research specialist with tools to retrieve company financial data, "
        "industry benchmarks, and competitor premium tier intelligence. "
        "Use me first when you need data to inform the analysis."
    ),
    system_prompt=(
        "You are a market research specialist in a strategic decision team. "
        "Use your tools to gather relevant data about the company, industry, and competitors. "
        "Summarize your findings clearly, then hand off to the analyst with the data."
    ),
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    callback_handler=None,
)

analyst = Agent(
    name="analyst",
    description=(
        "Business strategy analyst who evaluates options A/B/C. "
        "Use me after research is complete to get structured analysis of each option."
    ),
    system_prompt=(
        "You are a business strategy analyst. Evaluate each option (A, B, C) based on: "
        "strengths, weaknesses, implementation complexity (Low/Med/High), "
        "top 2 risks with mitigations, and a verdict. "
        "When done, hand off to the writer with your structured analysis."
    ),
    callback_handler=None,
)

writer = Agent(
    name="writer",
    description=(
        "Executive memo writer who produces the final leadership decision memo. "
        "Use me last, after research and analysis are complete."
    ),
    system_prompt=(
        "You are an executive memo writer. Synthesize all research and analysis into a "
        "leadership memo with: Recommendation, Options table (A/B/C), Top 3 Risks + mitigations, "
        "Success Metrics with targets, Decision Required (owner + deadline). "
        "This is the FINAL step — do NOT hand off to anyone else."
    ),
    callback_handler=None,
)

---

## Part 2 — Create and Run the Swarm

Unlike `GraphBuilder`, the Swarm has **no edges** — no fixed route is programmed. The Strands SDK automatically gives each agent a `handoff_to_agent` tool. Agents use it autonomously.

In [ ]:
swarm = Swarm(
    [researcher, analyst, writer],
    entry_point=researcher,     # first agent to receive the task
    max_handoffs=6,             # safety: max autonomous handoffs
    max_iterations=10,          # safety: max total agent iterations
    execution_timeout=180.0,    # hard timeout in seconds
    node_timeout=60.0,          # per-agent timeout
)

print("Running Swarm (no fixed path — agents decide autonomously)...")
t0 = time.time()
result = swarm(DECISION_BRIEF)
total_time = time.time() - t0

print(f"Status: {result.status} | {total_time:.1f}s")
print(f"Path that emerged: {[n.node_id for n in result.node_history]}")

---

## Part 3 — Inspect the Emerged Path and Final Output

In [ ]:
# The path that emerged — agents decided this autonomously
print("=== SWARM PATH ===")
for i, node in enumerate(result.node_history):
    print(f"  {i+1}. {node.node_id}")

print()
print("=== FINAL MEMO (writer output) ===")
print("-" * 60)
print(str(result.results["writer"]))
print("-" * 60)

In [ ]:
# Token usage across the full swarm
usage = result.accumulated_usage
print(f"{'Metric':<20} {'Value':>10}")
print("-" * 32)
print(f"{'Input tokens':<20} {usage.get('inputTokens', 0):>10}")
print(f"{'Output tokens':<20} {usage.get('outputTokens', 0):>10}")
print(f"{'Total tokens':<20} {usage.get('totalTokens', 0):>10}")
print(f"{'Agents in path':<20} {len(result.node_history):>10}")
print(f"{'Execution time':<20} {total_time:>9.1f}s")

---

## Part 4 — Swarm vs Sequential Chain

The swarm and the sequential chain (Module 2) both go: Researcher → Analyst → Writer.
The key difference: in Module 2 that path was **hardcoded in Python**. Here it **emerged at runtime**.

| | Sequential Chain (M2) | Dynamic Swarm (M5) |
|---|---|---|
| Path | Fixed: always R→A→W | Emergent: agents decide |
| Control | Python code | Agent autonomy |
| Flexibility | Low — change code to change path | High — change descriptions |
| Predictability | Deterministic | Non-deterministic |
| Best for | Known, stable workflows | Exploration, complex tasks |

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `Swarm([agents], entry_point=agent)` | Creates a swarm with no fixed routing |
| `description` field | How agents decide who to hand off to — write it for the model |
| `handoff_to_agent` | Automatically added to each agent by the SDK |
| `result.node_history` | The path that emerged at runtime |
| `result.results["agent_name"]` | Output from a specific agent |
| `result.accumulated_usage` | Total token usage across all agents |

---

## What's Next

In **Module 6: Agent-as-Tool (Capstone)**, you combine all four patterns into the complete Decision-Memo System from the deck: Parallel heads + Critic-Refiner + Agent-as-Tool specialists + Sequential synthesis.

---

> 💡 **Note — Memory & Observability (review):**
>
> **Memory:** The Swarm's shared context (handoff messages + previous agent outputs) is managed automatically by the SDK. Review: how does `invocation_state` interact with the Swarm's shared context? Can `AgentCoreMemorySessionManager` provide LTM across swarm sessions? (Module 6)
>
> **Observability:** With `callback_handler=None`, all 3 agents are invisible. In a production swarm with many agents, you'd want per-agent OTEL traces to see who handed off to whom and why. (Module 7)

---

## Run interactively

```bash
cd samples/05-dynamic-swarm
pip install -r requirements.txt
python chat.py
```